Sample of one page from dashboard. Currently working on cleaning it up and adding to github. Typically streamlit needs a .py file to run. Showing here in .ipynb format just for readability

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import numpy as np
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D

In [ ]:
# Tabs Setup
tab_titles = ["Home", "View Individual PFAS", "View summed PFAS", "View by location", "Learn More"]

# Track which tab is active
selected_tab = st.radio("Select View:", tab_titles, index=0)


In [ ]:
# sample of one page. first page is home page which preceeds this. It's mostly text which is simple.
# The 'view by location' and 'view individual PFAS' pages are more complex due to custom plotting
elif selected_tab == "View by location":
    with st.container():
        st.title("View by Location")
        st.write("Compare lab results across different locations with boxplots.")

        # Filters based on selected options
        location_options = complete["Location"].unique().tolist()
        selected_locations = st.multiselect("Select Location(s):", options=location_options, default=location_options)

        lab_result_options = complete["PFAS"].unique().tolist()
        selected_labs = st.multiselect("Select Lab Results:", options=lab_result_options, default=lab_result_options)

        year_options = complete["Year"].unique().tolist()
        year = st.multiselect("Select Year(s):",
                                    options = year_options,
                                    default=year_options,
                                    key = "year")

        # user must select one location, year, and result
        if not selected_labs or not selected_locations or not year:
            st.warning("Please select **at least one PFAS and location** to view the data.")

            with st.expander("Interested in viewing individual PFAS and compare results by location?"):
                st.write("Select **View Individual PFAS**. Then select the specific PFAS you are interested in viewing.")

            with st.expander("Interested in viewing  guidance for a combination of PFAS?"):
                st.write("Select **View summed PFAS**.")

            with st.expander("What does the NHANES 95th percentile line mean?"):
                st.write("The National Health and Nutrition Examination Survey (NHANES) is a national program of studies designed to assess the health and nutritional status of adults and children in the United States. This line represents the NHANES 95th percentile value for comparison. The 95th percentile line shows which communities have PFAS levels above the 95th percentile of the NHANES data.")
            st.stop()

In [ ]:
# this is part of the larger loop above. again, just using .ipynb for readibility but typically this would be one long script
# Define large and small range chemicals
    large_range = ["PFOS", "PFOA", "PFO5DoA"]
    small_range = ["PFHxS", "MeFOSAA", "Nafion byproduct 2"]
    medium_range = ["PFNA", "PFDA", "PFUnDA"]

    filtered_data = complete[(complete["Location"].isin(selected_locations)) & (complete["PFAS"].isin(selected_labs) & (complete["Year"].isin(year)))]

    # filtering only for locations and chemicals with detection rates above 50%
    # unable to plot boxplots for chemicals with detection rates below 50%
    detection_rates = filtered_data.groupby(['Year', "Location",  "PFAS"])['detect'].mean().reset_index()
    detection_rates.rename(columns={"detect": "detection_rate"}, inplace=True)
    locations_above_50_percent = detection_rates[detection_rates["detection_rate"] > 0.5]

    filtered_data = filtered_data.merge(locations_above_50_percent[['Year', "Location",  "PFAS"]], on=['Year', "Location",  "PFAS"], how='inner')

In [ ]:
    group_sizes = filtered_data.groupby(['Year', "Location",  "PFAS"]).size()
    valid_groups = group_sizes[group_sizes >= 5].index
    filtered_data_valid = filtered_data.set_index(['Year', "Location",  "PFAS"])
    filtered_data_valid = filtered_data_valid.loc[valid_groups].reset_index()

In [ ]:
    # typically boxplots use the min and max, but since there are outliers in the data, we use the 5th and 95th percentiles
    # have to manually calculate & update the data since seaborn doesn't support this
    # using seaborn over plotly (which would do this automatically) since seaborn allows us to set the boxplot width
    quantiles = filtered_data_valid.groupby(['Year', "Location",  "PFAS"])['conc'].quantile([0.05, 0.25, 0.5, 0.75, 0.95])
    quantiles_unstacked = quantiles.unstack()
    quantiles_unstacked.columns = ['5th percentile', 'lower', 'middle', 'upper', '95th percentile']

    df_plot_adjusted = filtered_data_valid.merge(quantiles_unstacked, on=['Year', "Location",  "PFAS"], how='left')
    df_plot_adjusted['conc_adjusted'] = df_plot_adjusted['conc']
    df_plot_adjusted['conc_adjusted'] = df_plot_adjusted['conc_adjusted'].clip(lower=df_plot_adjusted['5th percentile'], upper=df_plot_adjusted['95th percentile'])

The following 3 chunks of code are very similar. In order to get chemicals with similar ranges on the same row, I set up a grid, plot the large-range chemicals first, medium-range, and then small range. I will be turning the plots and formatting code into functions for less redudancy.

In [ ]:
# Grid setup
    num_cols = 3  # 3x3 grid
    cols = st.columns(num_cols)
    i = 0

    df_large = df_plot_adjusted[df_plot_adjusted['PFAS'].isin(large_range)]
    legend_added = False
    if not df_large.empty:
        for loc in selected_locations:
            df_plot = df_large[df_large['Location'] == loc]
            if not df_plot.empty:
                plotly_font = {"family": "Arial", "size": 15, "color": "#333"}
                fig, ax = plt.subplots(figsize=(6, 4), dpi = 400) =
                sns.boxplot(
                    data=df_plot,
                    x="Year",
                    y="conc_adjusted",
                    showfliers=False,
                    hue_order= predefined_large_order,
                    hue="PFAS",
                    palette=group_colors_pfas,
                    width=0.5,
                    dodge=True,
                    linewidth=1.5
                )
                ax.legend_.remove()

                # legend creates spacing issues for the other plots in the same row, so manually adding whitespace
                if len(df_large) > 0:
                    legend= fig.legend([], [], title=' ', loc='center left', bbox_to_anchor=(0, 1.15), ncol=3)
                    legend.get_frame().set_linewidth(0)
                    legend.get_frame().set_facecolor('none')

                if not legend_added:
                    fig.legend(title='PFAS', loc='upper left', bbox_to_anchor=(0, 1.15), ncol=9)
                    legend_added = True
                    fig.subplots_adjust(top=0.75)

                ax.set_title(loc, fontsize=16)
                ax.set_title(loc, fontsize=plotly_font["size"], fontfamily=plotly_font["family"], color=plotly_font["color"], fontweight="bold")
                ax.set_xlabel("Year", fontsize=plotly_font["size"], fontfamily=plotly_font["family"], color=plotly_font["color"], fontweight="bold")
                ax.set_ylabel("Concentration (ng/mL)", fontsize=plotly_font["size"], fontfamily=plotly_font["family"], color=plotly_font["color"], fontweight="bold")

                ax.set_ylim(0, df_large["conc_adjusted"].max() * 1.1)
                ax.yaxis.grid(True, linestyle="--", alpha=0.5)
                ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:1f}"))

                ax.xaxis.label.set_color("black")
                ax.yaxis.label.set_color("black")
                ax.tick_params(axis='x', labelsize=16)
                ax.tick_params(axis='y', labelsize=16)

                ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.1f}"))

                with cols[i % num_cols]:
                    st.pyplot(fig)

                i += 1

In [ ]:

    cols = st.columns(num_cols)
    i = 0
    df_small = df_plot_adjusted[df_plot_adjusted['PFAS'].isin(small_range)]

    if not df_small.empty:
        for loc in selected_locations:
            df_plot = df_small[df_small['Location'] == loc]
            if not df_plot.empty:
                plotly_font = {"family": "Arial", "size": 16, "color": "black"}
                fig, ax = plt.subplots(figsize=(6, 4), dpi = 400)

                sns.boxplot(
                    data=df_plot,
                    x="Year",
                    y="conc_adjusted",
                    showfliers=False,
                    hue_order= predefined_small_order,
                    hue="PFAS",
                    palette=group_colors_pfas,
                    width=0.5,
                    dodge=True,
                    linewidth=1.5
                )
                ax.legend_.remove()
                if not legend_added:
                    fig.legend(title='PFAS', loc='upper left', bbox_to_anchor=(0, 1.15), ncol=9)
                    legend_added = True
                    fig.subplots_adjust(top=0.75)

                ax.set_title(loc, fontsize=16)
                ax.set_title(loc, fontsize=17, fontfamily=plotly_font["family"], color=plotly_font["color"], fontweight="bold")
                ax.set_xlabel("Year", fontsize=17, fontfamily=plotly_font["family"], color=plotly_font["color"], fontweight="bold")
                ax.set_ylabel("Concentration (ng/mL)", fontsize=17, fontfamily=plotly_font["family"], color=plotly_font["color"], fontweight="bold")
                ax.set_ylim(0, df_small["conc_adjusted"].max() * 1.1)
                ax.yaxis.grid(True, linestyle="--", alpha=0.5)
                ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:1f}"))

                ax.xaxis.label.set_color("black")
                ax.yaxis.label.set_color("black")
                ax.tick_params(axis='x', labelsize=17)
                ax.tick_params(axis='y', labelsize=17)

                ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.1f}"))

                with cols[i % num_cols]:
                    st.pyplot(fig)

                i += 1

In [ ]:
    cols = st.columns(num_cols)
    i = 0
    df_medium = df_plot_adjusted[df_plot_adjusted['PFAS'].isin(medium_range)]
    if not df_medium.empty:
        for loc in selected_locations:
            df_plot_med = df_medium[df_medium['Location'] == loc]
            if not df_plot_med.empty:
                plotly_font = {"family": "Arial", "size": 16, "color": "black"}
                fig, ax_1 = plt.subplots(figsize=(6, 4), dpi = 400)

                sns.boxplot(
                    data=df_plot_med,
                    x="Year",
                    y="conc_adjusted",
                    showfliers=False,
                    hue="PFAS",
                    hue_order= predefined_medium_order,
                    palette=group_colors_pfas,
                    width=0.5,  # fixed box width
                    dodge=True,
                    linewidth=1.5
                )

                ax_1.legend_.remove()
                # only want the legend to show once
                if not legend_added:
                    fig.legend(title='PFAS', loc='upper left', bbox_to_anchor=(0, 1.15), ncol=9)
                    legend_added = True
                    fig.subplots_adjust(top=0.75)

                ax_1.set_title(loc, fontsize=16)

                ax_1.set_title(loc, fontsize=plotly_font["size"], fontfamily=plotly_font["family"], color=plotly_font["color"], fontweight="bold")
                ax_1.set_xlabel("Year", fontsize=plotly_font["size"], fontfamily=plotly_font["family"], color=plotly_font["color"], fontweight="bold")
                ax_1.set_ylabel("Concentration (ng/mL)", fontsize=plotly_font["size"], fontfamily=plotly_font["family"], color=plotly_font["color"], fontweight="bold")

                ax_1.set_ylim(0, df_medium["conc_adjusted"].max() * 1.1)
                ax_1.yaxis.grid(True, linestyle="--", alpha=0.5)
                ax_1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:1f}"))

                ax_1.xaxis.label.set_color("black")

                ax_1.tick_params(axis='x', labelsize=17)

                with cols[i % num_cols]:
                    st.pyplot(fig)

                i += 1

In [ ]:
# infographic/text boxes for each selected chemical below the plot
title = f"General {selected_labs} Overview"
columns_per_row = 3

# Loop through the selected lab_results in batches of 3 and display in columns
for i in range(0, len(selected_labs), columns_per_row):
    col_batch = st.columns(columns_per_row)
    for j, lab_result in enumerate(selected_labs[i:i + columns_per_row]):
        with col_batch[j]:
            facts = f"""
            <ul>
                <li><strong>Fact for {lab_result}</strong> Specific to this lab result.</li>
                <li>Additional details about {lab_result}.</li>
                <li>Some other interesting fact about {lab_result}.</li>
            </ul>
            """
            st.markdown(f"""
                <div style="background-color: #f9f9f9; padding: 10px; border-radius: 5px; border: 1px solid #e1e1e1;">
                    <h4 style="color: #333;">Information for {lab_result}</h4>
                    {facts}
                </div>
            """, unsafe_allow_html=True)